In [ ]:
import kagglehub

In [ ]:
# Download latest version
path = kagglehub.dataset_download("thisisjibon/banglabeats3sec")

print("Path to dataset files:", path)

In [3]:
from pathlib import Path
import shutil, random, csv
from math import floor

In [ ]:
# SOURCE_DIR = Path("D:/MSc/3. Spring 2026/CSE715/Project/wavs3sec/") # -> DO NOT REMOVE
SOURCE_DIR = Path(input("Enter the path where the above Kaggle dataset was downloaded in your device: "))
DEST_DIR = Path("../data/audio/bn/")
METADATA_FILE = Path("./metadata_bn.csv")

In [15]:
def find_audio_files(subdir):
    return [f for f in subdir.iterdir() if f.is_file() and f.suffix.lower() == ".wav"]

In [37]:
def transfer_audio_files(files, subdir, metadata_rows, copy_instead_of_move=True):
    for src_file in files:
        dst_file = DEST_DIR / src_file.name
        if dst_file.exists():
            stem = dst_file.stem
            suffix = dst_file.suffix
            counter = 1
            while True:
                new_name = f"{stem}_{counter}{suffix}"
                candidate = DEST_DIR / new_name
                if not candidate.exists():
                    dst_file = candidate
                    break
                counter += 1
        shutil.copy2(src_file, dst_file) if copy_instead_of_move else shutil.move(str(src_file), str(dst_file))
        metadata_rows.append({
            "label": dst_file.name,
            "genre": subdir.name
        })
        
    return metadata_rows

In [6]:
random.seed(42)

In [9]:
DEST_DIR.mkdir(parents=True, exist_ok=True)

In [38]:
metadata_rows = []

In [39]:
k = 10

In [40]:
total = 0

In [42]:
for subdir in SOURCE_DIR.iterdir():
    try:
        if not subdir.is_dir(): continue
        files = find_audio_files(subdir=subdir)
        n = len(files)
        if n == 0: continue
        t = floor((k / 100) * n)
        if k > 0 and t == 0: t = 1
        t = min(t, n)
        print(t)
        total += t
        chosen_files = random.sample(files, t)
        metadata_rows = transfer_audio_files(files=chosen_files, subdir=subdir, metadata_rows=metadata_rows)
    except Exception as e:
        print(e)

223
235
173
202
205
192
196
191


In [43]:
total

1617

In [46]:
with open(METADATA_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["label", "genre"])
    writer.writeheader()
    writer.writerows(metadata_rows)